## dev note
use azureml_py38 (python 3.10.11)

In [ ]:
%pip install stable-baselines3 shimmy gymnasium

In [ ]:
import mlflow
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from environment_v3 import CancerSimulation
from analyzer_v3 import PatientAnalyzer
import os
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.callbacks import CallbackList

# This "Reporter" sends the AI's score to Azure every step
class AzureMetricCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(AzureMetricCallback, self).__init__(verbose)

    def _on_step(self) -> bool:
        # Log the reward to create the chart
        if 'rewards' in self.locals:
            mlflow.log_metric("reward", self.locals['rewards'][0], step=self.num_timesteps)
        return True

# --- Start Training ---
mlflow.set_experiment("Peacekeeper_Evolutionary_Trap")

with mlflow.start_run():
    # Load profile
    analyzer = PatientAnalyzer("data/Gene_Expression_Analysis_and_Disease_Relationship_Synthetic.csv")
    profile = analyzer.get_strategic_profile()
    print("initial_tumor_size: ", profile['initial_tumor_size'])
    
    # Log these as "Facts" (Overview)
    mlflow.log_param("patient_growth_rate", profile['avg_growth'])
    mlflow.log_param("max_resistance", profile['max_res_a'])

    # Setup Env
    env = CancerSimulation(profile)
    model = PPO("MlpPolicy", env, verbose=1)

    # 1. Define the save directory
    save_dir = "./loaded_models"

    # Optional but recommended: Create the folder if it doesn't exist yet
    os.makedirs(save_dir, exist_ok=True)

    # 2. Set up the CheckpointCallback
    # save_freq: How often to save the model (in steps, not episodes)
    # save_path: Where to save the models
    # name_prefix: The prefix for your saved files
    checkpoint_callback = CheckpointCallback(
        save_freq=100,
        save_path=save_dir,
        name_prefix="oncosteer_v3_model"
    )
    callback_list = CallbackList([AzureMetricCallback(), checkpoint_callback])

    print("🚀 Training with live metrics... Check the 'Metrics' tab in Azure!")
    
    # We pass the callback here
    model.learn(total_timesteps=100000, callback=callback_list)

    # Save the result
    model.save("peacekeeper_metric_model")
    mlflow.log_artifact("peacekeeper_metric_model.zip")

In [ ]:
model.save("peacekeeper_final_azure_v3")
print("Model saved successfully!")